# Development of `StateVector` object

In [1]:
import keyword 
import math 
from numbers import Real
import keyword

from dataclasses import dataclass

import numpy as np

# 1) Helper Functions

- `_validate_text`
- `_validate_name`
- `_validate_scalar`

## 1.1 - `_validate_text`

In [2]:
def _validate_text(value:object, label:str) -> str:
    """
    Validate and return non-empty text.

    Parameters
    ----------
    value : object
        Input value to validate.

    label : str
        Human-readable name used in error messages.

    Returns
    -------
    str
        Input text with leading and trailing whitespace removed.

    Example
    -------
    >>> _validate_text("  m/s  ", "StateField unit")
    'm/s'
    """
    if not isinstance(value, str):
        raise TypeError(
            f"{label} must be a string; received {type(value).__name__}"
        )

    cleaned = value.strip()

    if cleaned == "":
        raise ValueError(
            f"{label} must be a non-empty string"
        )

    return cleaned

In [3]:
# Example Usage:
print(_validate_text("123   ", "StateField unit"))
#print(_validate_text(123, "StateField unit"))

123


## 1.2 - `_validate_name`

In [4]:
def _validate_name(value: object, label: str) -> str:
    """
    Validate and return a symbolic name.

    Parameters
    ----------
    value : object
        Input value to validate.

    label : str
        Human-readable name used in error messages.

    Returns
    -------
    str
        Validated name with leading and trailing whitespace removed.

    Example
    -------
    >>> _validate_name("  x_dot  ", "StateField name")
    'x_dot'
    """
    name = _validate_text(value, label)

    if not name.isidentifier():
        raise ValueError(
            f"{label} must be a valid Python identifier; received {name!r}."
        )

    if keyword.iskeyword(name):
        raise ValueError(
            f"{label} cannot be a Python keyword; received {name!r}."
        )

    return name

In [5]:
# Example Usage:
#_validate_name("m/s", "StateField name")
_validate_name("x_dot", "StateField name")

'x_dot'

## 1.3 - `_validate_scalar`

In [6]:
def _validate_scalar(value: object, label: str) -> float:
    """
    Validate and return a finite real scalar.

    Parameters
    ----------
    value : object
        Input value to validate.

    label : str
        Human-readable name used in error messages.

    Returns
    -------
    float
        Validated scalar converted to float.

    Example
    -------
    >>> _validate_scalar(10, "StateField value")
    10.0
    """
    if isinstance(value, bool):
        raise TypeError(
            f"{label} must be a finite real scalar; boolean values are not allowed."
        )

    if not isinstance(value, Real):
        raise TypeError(
            f"{label} must be a finite real scalar; received {type(value).__name__}."
        )

    scalar = float(value)

    if not math.isfinite(scalar):
        raise ValueError(
            f"{label} must be finite; received {scalar}."
        )

    return scalar

In [7]:
# Example Usage:
#_validate_scalar(True, "x_dot")
_validate_scalar(1, "x_dot")

1.0

# 2) `StateField`

In [8]:
@dataclass(frozen=True)
class StateField:
    """
    To insantiate One scalar component of a state vector in the StateVector class.

    Parameters
    ----------
    name : str
        Symbolic name of the state component.

    value : float
        Numerical value of the state component.

    unit : str
        Unit associated with the state component.

    Usage
    -----
    >>> StateField("x", 10.0, "m")
    StateField(name='x', value=10.0, unit='m')
    """

    name: str
    value: float
    unit: str

    def __post_init__(self) -> None:
        name = _validate_name(self.name, "StateField name")

        value = _validate_scalar(
            self.value,
            f"Value for state field {name!r}",
        )

        unit = _validate_text(
            self.unit,
            f"Unit for state field {name!r}",
        )

        object.__setattr__(self, "name", name)
        object.__setattr__(self, "value", value)
        object.__setattr__(self, "unit", unit)

In [9]:
# Example Usage:
field = StateField("  x  ", 10, "  m  ")

print(field)
print(field.name)
print(field.value)
print(field.unit)


StateField(name='x', value=10.0, unit='m')
x
10.0
m


# 3) `StateVector`

In [10]:
class StateVector:
    """
    Ordered state vector with fixed layout and mutable values.

    Parameters
    ----------
    **fields
        State components specified as keyword arguments. Each field may be
        written as either a scalar value or a ``(value, unit)`` tuple.

    Properties
    ----------
    dimension
        Number of scalar state components.

    names
        Ordered state component names.

    units
        Ordered state component units.

    values
        Copy of the current numerical state vector.

    fields
        Current state components as StateField objects.

    Methods
    -------
    index(name)
        Return the integer index of a named state component.

    unit(name)
        Return the unit of a named state component.

    set(name, value)
        Update one named state component in place.

    set_values(values)
        Update the full numerical state vector in place.

    with_values(values)
        Create a new StateVector with the same layout and new values.

    summary()
        Print a compact state-vector summary.

    Usage
    -----
    >>> state = StateVector(x=(1.0, "m"), vx=(0.0, "m/s"))
    >>> state["x"]
    1.0
    >>> state.set("x", 2.0)
    >>> state.values
    array([2., 0.])
    """

    # ------------------------------------------------------------------
    # Construction
    # ------------------------------------------------------------------

    def __init__(self, **fields: object) -> None:
        if len(fields) == 0:
            raise ValueError("StateVector must contain at least one state field.")

        parsed_fields = tuple(
            self._parse_field(name, raw_value)
            for name, raw_value in fields.items()
        )

        names = tuple(field.name for field in parsed_fields)
        units = tuple(field.unit for field in parsed_fields)
        values = np.array(
            [field.value for field in parsed_fields],
            dtype=float,
        )

        if len(set(names)) != len(names):
            raise ValueError(
                "StateVector component names must be unique after validation."
            )

        self._names = names
        self._units = units
        self._values = values

        self._name_to_index = {
            name: index
            for index, name in enumerate(self._names)
        }

    # ------------------------------------------------------------------
    # Properties
    # ------------------------------------------------------------------

    @property
    def dimension(self) -> int:
        """Return the number of scalar state components."""
        return len(self._names)

    @property
    def names(self) -> tuple[str, ...]:
        """Return ordered state component names."""
        return self._names

    @property
    def units(self) -> tuple[str, ...]:
        """Return ordered state component units."""
        return self._units

    @property
    def values(self) -> np.ndarray:
        """Return a copy of the current numerical state vector."""
        return self._values.copy()

    @property
    def fields(self) -> tuple[StateField, ...]:
        """Return current state components as StateField objects."""
        return tuple(
            StateField(name, value, unit)
            for name, value, unit in zip(
                self._names,
                self._values,
                self._units,
            )
        )

    # ------------------------------------------------------------------
    # Public lookup/access methods
    # ------------------------------------------------------------------

    def index(self, name: str) -> int:
        """
        Return the integer index of a named state component.

        Parameters
        ----------
        name : str
            State component name.

        Returns
        -------
        int
            Integer index of the component in the state vector.

        Usage
        -----
        >>> state = StateVector(x=(1.0, "m"), vx=(0.0, "m/s"))
        >>> state.index("vx")
        1
        """
        name = _validate_name(name, "State component name")

        if name not in self._name_to_index:
            raise KeyError(
                f"Unknown state component {name!r}. "
                f"Available components are {self._names}."
            )

        return self._name_to_index[name]

    def unit(self, name: str) -> str:
        """
        Return the unit of a named state component.

        Parameters
        ----------
        name : str
            State component name.

        Returns
        -------
        str
            Unit string associated with the state component.

        Usage
        -----
        >>> state = StateVector(x=(1.0, "m"))
        >>> state.unit("x")
        'm'
        """
        return self._units[self.index(name)]

    # ------------------------------------------------------------------
    # Public mutation methods
    # ------------------------------------------------------------------

    def set(self, name: str, value: object) -> None:
        """
        Update one named state component in place.

        Parameters
        ----------
        name : str
            State component name.

        value : object
            New scalar value.

        Returns
        -------
        None

        Usage
        -----
        Useful for manual edits, debugging, or interactive notebooks.

        >>> state.set("x", 10.0)
        """
        index = self.index(name)

        scalar = _validate_scalar(
            value,
            f"Value for state component {name!r}",
        )

        self._values[index] = scalar

    def set_values(self, values: object) -> None:
        """
        Update the full numerical state vector in place.

        Parameters
        ----------
        values : object
            New numerical values with shape ``(dimension,)``.

        Returns
        -------
        None

        Usage
        -----
        Used during filtering after prediction or correction.

        >>> x_pred = F @ state.values
        >>> state.set_values(x_pred)
        """
        self._values[:] = self._coerce_values(
            values,
            self.dimension,
            "StateVector values",
        )

    # ------------------------------------------------------------------
    # Public copy/snapshot methods
    # ------------------------------------------------------------------

    def with_values(self, values: object) -> "StateVector":
        """
        Create a new StateVector with the same layout and new values.

        Parameters
        ----------
        values : object
            New numerical values with shape ``(dimension,)``.

        Returns
        -------
        StateVector
            New state vector with the same names and units.

        Usage
        -----
        Useful for snapshots or what-if calculations without mutating the
        current StateVector.
        """
        new_values = self._coerce_values(
            values,
            self.dimension,
            "StateVector values",
        )

        field_inputs = {
            name: (value, unit)
            for name, value, unit in zip(
                self._names,
                new_values,
                self._units,
            )
        }

        return StateVector(**field_inputs)

    # ------------------------------------------------------------------
    # Python protocol methods
    # ------------------------------------------------------------------

    def __getitem__(self, key: str | int | slice) -> float | np.ndarray:
        """
        Return state values by name, integer index, or slice.

        Parameters
        ----------
        key : str, int, or slice
            Name, integer index, or slice used to access state values.

        Returns
        -------
        float or np.ndarray
            Selected state value or copied array slice.

        Usage
        -----
        >>> state["x"]
        >>> state[0]
        >>> state[0:3]
        """
        if isinstance(key, str):
            return float(self._values[self.index(key)])

        if isinstance(key, bool):
            raise TypeError("Boolean indexing is not supported for StateVector.")

        if isinstance(key, int):
            return float(self._values[key])

        if isinstance(key, slice):
            return self._values[key].copy()

        raise TypeError(
            "StateVector indices must be a state name, integer index, or slice; "
            f"received {type(key).__name__}."
        )

    def __len__(self) -> int:
        """Return the state-vector dimension."""
        return self.dimension

    # ------------------------------------------------------------------
    # Display/debug methods
    # ------------------------------------------------------------------

    def summary(self) -> None:
        """
        Print a compact state-vector summary.

        Parameters
        ----------
        None

        Returns
        -------
        None

        Usage
        -----
        Used for quick inspection of state ordering, values, and units.
        """
        print("StateVector")
        print(f"  dimension: {self.dimension}")

        for index, name in enumerate(self._names):
            value = self._values[index]
            unit = self._units[index]
            print(f"  [{index}] {name} = {value} [{unit}]")

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _parse_field(name: str, raw_value: object) -> StateField:
        """
        Parse one keyword argument into a StateField.

        Parameters
        ----------
        name : str
            State component name.

        raw_value : object
            Either a scalar value or a ``(value, unit)`` tuple.

        Returns
        -------
        StateField
            Validated scalar state field.

        Usage
        -----
        Used internally by ``StateVector.__init__`` to convert inputs such as
        ``x=(1.0, "m")`` or ``x=1.0`` into canonical StateField objects.
        """
        field_name = _validate_name(name, "StateField name")

        if isinstance(raw_value, tuple):
            if len(raw_value) != 2:
                raise ValueError(
                    f"State field {field_name!r} must be specified as "
                    "a scalar value or a (value, unit) tuple."
                )

            raw_scalar, raw_unit = raw_value

            value = _validate_scalar(
                raw_scalar,
                f"Value for state field {field_name!r}",
            )
            unit = _validate_text(
                raw_unit,
                f"Unit for state field {field_name!r}",
            )

            return StateField(field_name, value, unit)

        value = _validate_scalar(
            raw_value,
            f"Value for state field {field_name!r}",
        )

        return StateField(field_name, value, "unspecified")

    @staticmethod
    def _coerce_values(
        values: object,
        dimension: int,
        label: str,
    ) -> np.ndarray:
        """
        Validate and return a finite numerical vector.

        Parameters
        ----------
        values : object
            Input values to validate.

        dimension : int
            Expected vector length.

        label : str
            Human-readable name used in error messages.

        Returns
        -------
        np.ndarray
            Validated one-dimensional float array.

        Usage
        -----
        Used internally before replacing the numerical values of the state.
        """
        candidate = np.asarray(values, dtype=object)

        if candidate.shape != (dimension,):
            raise ValueError(
                f"{label} must have shape ({dimension},); "
                f"received shape {candidate.shape}."
            )

        return np.array(
            [
                _validate_scalar(value, f"{label}[{index}]")
                for index, value in enumerate(candidate)
            ],
            dtype=float,
        )

In [11]:
# Example Usage:

# The keyword argument order defines the numerical state ordering.
state = StateVector(
    x=(1.0, "m"),
    y=(2.0, "m"), 
    z=(3.0, "m"),
    vx=(0.0, "m/s"),
    vy=(0.0, "m/s"),
    vz=(0.0, "m/s"),
    wx=(0.0, "rad/s"),
    wy=(0.0, "rad/s"),
    wz=(0.0, "rad/s"),
    q1=(1.0, "unitless"),
    q2=(0.0, "unitless"),
    q3=(0.0, "unitless"),
    q4=(0.0, "unitless")
)

print("Initial state:")
state.summary()

Initial state:
StateVector
  dimension: 13
  [0] x = 1.0 [m]
  [1] y = 2.0 [m]
  [2] z = 3.0 [m]
  [3] vx = 0.0 [m/s]
  [4] vy = 0.0 [m/s]
  [5] vz = 0.0 [m/s]
  [6] wx = 0.0 [rad/s]
  [7] wy = 0.0 [rad/s]
  [8] wz = 0.0 [rad/s]
  [9] q1 = 1.0 [unitless]
  [10] q2 = 0.0 [unitless]
  [11] q3 = 0.0 [unitless]
  [12] q4 = 0.0 [unitless]


In [12]:
print("\nProperties:")
print("dimension:", state.dimension)
print("names:", state.names)
print("units:", state.units)
print("values:", state.values)
print("number of fields:", len(state.fields))


Properties:
dimension: 13
names: ('x', 'y', 'z', 'vx', 'vy', 'vz', 'wx', 'wy', 'wz', 'q1', 'q2', 'q3', 'q4')
units: ('m', 'm', 'm', 'm/s', 'm/s', 'm/s', 'rad/s', 'rad/s', 'rad/s', 'unitless', 'unitless', 'unitless', 'unitless')
values: [1. 2. 3. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
number of fields: 13


In [13]:
print("Lookup / access:")
print("index of x:", state.index("x"))
print("index of vz:", state.index("vz"))
print("unit of vx:", state.unit("vx"))
print("unit of q1:", state.unit("q1"))


Lookup / access:
index of x: 0
index of vz: 5
unit of vx: m/s
unit of q1: unitless


In [14]:
print("\nPython-style access:")
print('state["x"]:', state["x"])
print('state["q1"]:', state["q1"])
print("state[0]:", state[0])
print("state[0:3]:", state[0:3])
print("len(state):", len(state))


Python-style access:
state["x"]: 1.0
state["q1"]: 1.0
state[0]: 1.0
state[0:3]: [1. 2. 3.]
len(state): 13


In [15]:
print("\nUpdate one scalar component:")
state.set("x", 10.0)
state.set("vx", 0.5)

print('state["x"] after set:', state["x"])
print('state["vx"] after set:', state["vx"])



Update one scalar component:
state["x"] after set: 10.0
state["vx"] after set: 0.5


In [16]:
print("Update full state vector:")

new_values = state.values
new_values[state.index("y")] = 20.0
new_values[state.index("vy")] = 1.5

state.set_values(new_values)

print('state["y"] after set_values:', state["y"])
print('state["vy"] after set_values:', state["vy"])


Update full state vector:
state["y"] after set_values: 20.0
state["vy"] after set_values: 1.5


In [17]:
print("\nKF-style usage pattern:")

dt = 1.0

# Simple example: position update using velocity.
# This is not a full Kalman filter yet. It only demonstrates how
# StateVector values can be updated after computing a predicted state.
predicted_values = state.values

predicted_values[state.index("x")] += dt * state["vx"]
predicted_values[state.index("y")] += dt * state["vy"]
predicted_values[state.index("z")] += dt * state["vz"]

state.set_values(predicted_values)

print("State after simple prediction step:")
state.summary()


KF-style usage pattern:
State after simple prediction step:
StateVector
  dimension: 13
  [0] x = 10.5 [m]
  [1] y = 21.5 [m]
  [2] z = 3.0 [m]
  [3] vx = 0.5 [m/s]
  [4] vy = 1.5 [m/s]
  [5] vz = 0.0 [m/s]
  [6] wx = 0.0 [rad/s]
  [7] wy = 0.0 [rad/s]
  [8] wz = 0.0 [rad/s]
  [9] q1 = 1.0 [unitless]
  [10] q2 = 0.0 [unitless]
  [11] q3 = 0.0 [unitless]
  [12] q4 = 0.0 [unitless]


In [18]:

print("\nCreate a separate snapshot with different values:")

snapshot_values = state.values
snapshot_values[state.index("x")] = 999.0

snapshot = state.with_values(snapshot_values)

print('original state["x"]:', state["x"])
print('snapshot["x"]:', snapshot["x"])


Create a separate snapshot with different values:
original state["x"]: 10.5
snapshot["x"]: 999.0


In [19]:
print("\nSafety check:")

external_values = state.values
external_values[state.index("x")] = -999.0

print("Modified external_values only.")
print('state["x"] is unchanged:', state["x"])


Safety check:
Modified external_values only.
state["x"] is unchanged: 10.5


In [20]:
print("Final state:")
state.summary()

Final state:
StateVector
  dimension: 13
  [0] x = 10.5 [m]
  [1] y = 21.5 [m]
  [2] z = 3.0 [m]
  [3] vx = 0.5 [m/s]
  [4] vy = 1.5 [m/s]
  [5] vz = 0.0 [m/s]
  [6] wx = 0.0 [rad/s]
  [7] wy = 0.0 [rad/s]
  [8] wz = 0.0 [rad/s]
  [9] q1 = 1.0 [unitless]
  [10] q2 = 0.0 [unitless]
  [11] q3 = 0.0 [unitless]
  [12] q4 = 0.0 [unitless]
